In [1]:
import os
import yaml
import warnings; warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from byaldi import RAGMultiModalModel
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

In [2]:
load_dotenv(dotenv_path="../.env") 

with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

env_var_name = config['api_keys']['google_gemini_env_name']
gemini_api_key = os.getenv(env_var_name)

if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    print("✅ Gemini API 키가 성공적으로 로드되었습니다.")
else:
    print("❌ API 키를 찾을 수 없습니다. .env 파일을 확인해주세요.")

✅ Gemini API 키가 성공적으로 로드되었습니다.


In [3]:
print("⏳ ColPali 비전 모델(vidore/colpali-v1.2)을 불러오는 중...")
vision_retriever = RAGMultiModalModel.from_pretrained("vidore/colpali-v1.2")
print("✅ 모델 로드 완료!")

⏳ ColPali 비전 모델(vidore/colpali-v1.2)을 불러오는 중...
Verbosity is set to 1 (active). Pass verbose=0 to make quieter.


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 413.68it/s]
The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


✅ 모델 로드 완료!


In [4]:
data_dir = "../data/raw/" # 10개의 PDF가 들어있는 폴더 경로
index_name = "multi_doc_vision_index"

print(f"📚 '{data_dir}' 내의 모든 PDF를 고화질 이미지 벡터로 변환하여 인덱싱합니다...")
print("⚠️ 주의: 다중 문서 환경이므로 시간이 꽤 소요될 수 있습니다 (GPU 권장).")

# byaldi는 폴더 경로를 주면 폴더 안의 모든 PDF를 한 번에 처리합니다.
vision_retriever.index(
    input_path=data_dir,
    index_name=index_name,
    store_collection_with_index=True,
    overwrite=True
)
print("✅ 다중 문서 ColPali Vision DB 구축 완료!")

📚 '../data/raw/' 내의 모든 PDF를 고화질 이미지 벡터로 변환하여 인덱싱합니다...
⚠️ 주의: 다중 문서 환경이므로 시간이 꽤 소요될 수 있습니다 (GPU 권장).
overwrite is on. Deleting existing index multi_doc_vision_index to build a new one.
Indexing file: ..\data\raw\hanmi_semiconductor_report_2026.pdf
Added page 1 of document 0 to index.
Added page 2 of document 0 to index.
Added page 3 of document 0 to index.
Added page 4 of document 0 to index.
Added page 5 of document 0 to index.
Added page 6 of document 0 to index.
Added page 7 of document 0 to index.
Added page 8 of document 0 to index.
Added page 9 of document 0 to index.
Added page 10 of document 0 to index.
Added page 11 of document 0 to index.
Added page 12 of document 0 to index.
Added page 13 of document 0 to index.
Added page 14 of document 0 to index.
Added page 15 of document 0 to index.
Added page 16 of document 0 to index.
Added page 17 of document 0 to index.
Added page 18 of document 0 to index.
Added page 19 of document 0 to index.
Added page 20 of document 0 to index.
A

KeyboardInterrupt: 

In [ ]:
print("\n" + "="*50)
print("🧐 AI에게 어려운 질문을 던져봅시다 (시각적 단절 테스트)")

# Baseline과 완전히 동일한 구체적인 질문으로 공정하게 비교(Ablation Study)합니다.
query = "삼성전자 2024년 사업보고서에서, 연결재무제표 기준 삼성전자의 당기순이익은 얼마인가요?"
print(f"질문: {query}\n")

# 비전 검색: 가장 유사한 1장의 '페이지 이미지'만 가져옴 (시각적 맹점 발생 원인!)
print("🔍 ColPali로 질문과 가장 유사한 '단일 페이지 이미지' 1장을 찾습니다...")
results = vision_retriever.search(query, k=1)

best_page_base64 = results[0].base64
best_page_num = results[0].page_num
best_doc_name = results[0].doc_id # 다중 문서 중 어떤 문서를 가져왔는지 확인

print(f"🎯 검색 완료: [{best_doc_name}]의 {best_page_num}페이지가 가장 관련성이 높습니다.\n")

# Gemini 2.5 Flash를 로드하여 이미지 해석 지시
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

message = HumanMessage(
    content=[
        {
            "type": "text", 
            "text": f"당신은 꼼꼼한 금융 데이터 분석가입니다.\n아래 제공된 [단일 문서 이미지]를 직접 눈으로 확인하고 사용자의 [질문]에 답변해주세요.\n이미지 안의 표가 잘려 있거나 내용이 없다면 억지로 유추하지 말고 '제공된 이미지에서 명확한 수치를 찾기 어렵습니다.'라고 답변하세요.\n\n[질문]\n{query}"
        },
        {
            "type": "image_url", 
            "image_url": {"url": f"data:image/jpeg;base64,{best_page_base64}"}
        }
    ]
)

print("🧠 Gemini 2.5 Flash가 제공된 1장의 이미지를 눈으로 읽고 분석 중입니다...")
response = llm.invoke([message])

print("=" * 50)
print("🤖 최종 AI 답변 (Vision-only RAG):")
print(response.content)
print("=" * 50)

It is a prefill stage but The `token_type_ids` is not provided. We recommend passing `token_type_ids` to the model to prevent bad attention masking.



🧐 AI에게 어려운 질문을 던져봅시다 (시각적 단절 테스트)
질문: 삼성전자 2024년 사업보고서에서, 연결재무제표 기준 삼성전자의 당기순이익은 얼마인가요?

🔍 ColPali로 질문과 가장 유사한 '단일 페이지 이미지' 1장을 찾습니다...
🎯 검색 완료: [0]의 195페이지가 가장 관련성이 높습니다.

🧠 Gemini 2.5 Flash가 제공된 1장의 이미지를 눈으로 읽고 분석 중입니다...
🤖 최종 AI 답변 (Vision-only RAG):
제공된 이미지에서 삼성전자의 연결재무제표 기준 당기순이익을 명확하게 찾기 어렵습니다.

이미지에 제시된 표는 "(2) 주요 관계기업"의 요약 재무상태표 및 요약 포괄손익계산서로, "삼성전자(주)"를 포함한 여러 관계기업의 개별 재무정보를 요약하여 보여주고 있습니다. 이는 삼성전자 자체의 연결재무제표가 아닌, 관계기업별 정보를 나타내는 것으로 보입니다.

따라서, 질문하신 '연결재무제표 기준 삼성전자의 당기순이익'에 해당하는 명확한 수치를 이 이미지에서 찾을 수 없습니다.
